# EuroSAT: NoLA full parameter grid
Use **Robotics Research Lab**. Set `run_limit` to a number of jobs or `None` for all; the default is one. The grid covers the listed values only. Keep GPU notebooks you are not using shut down to free memory, after saving your work. Validation is shared; the test split is not scored. Backbones, transforms, optimizers, head learning rate (0.001) and label smoothing stay fixed.

## 1. Configure the experiment
An epoch is one pass over the relevant training examples: actual anchors for the DINO head, whole adaptation subset for prompted CLIP. Each batch produces one update; the last partial batch is kept, so `updates_per_epoch = ceil(examples / batch_size)`. The smaller subset is a prefix of the larger one. Batch-size comparisons at the same epoch count match image exposure, not optimizer updates. Epoch 1 and epoch 4 are evaluations along one CLIP run, not separate trainings.

In [ ]:
from pathlib import Path
import itertools, json, random, sys
import pandas as pd

import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from IPython.display import display

root = Path.home() / "nola-walkthrough"
upstream = root / "upstream"
sys.dont_write_bytecode = True
sys.path[:0] = [str(upstream), str(root / ".deps")]
from clip import clip
from utils.data_utils import ds_specific_templates
from utils.model_utils import te_transform, tr_transforms, top_k_indices_per_class2


grid = dict(train_n=[160, 320], anchor_cap=[4, 16], head_batch=[8, 32],
            head_epochs=[3, 20], prompts=[16, 50], clip_batch=[4, 8], clip_lr=[0.001, 0.004])
clip_epochs = [1, 4]
seed, val_seed, val_n = 42, 123, 240
inference_batch = 4
head_lr = 0.001
clip_label_smoothing = 0.1
clip_backbone = "ViT-B/32"
dataset_name = "EuroSAT"
run_limit = 1
results_path = root / "eurosat-full-sweep.jsonl"
jobs = [dict(zip(grid, values)) for values in itertools.product(*grid.values())]
image_dir = root / "data/eurosat/EuroSAT_RGB"
split = json.loads((image_dir.parent / "split_zhou_EuroSAT.json").read_text())
name_by_id = {label: name for _, label, name in split["train"]}
names = [name_by_id[i] for i in sorted(name_by_id)]
pool = random.Random(seed).sample(split["train"], max(grid["train_n"]))
val = random.Random(val_seed).sample(split["val"], val_n)
audit = torch.tensor([row[1] for row in pool])
y_val = torch.tensor([row[1] for row in val])


def load_images(rows, transform):
    batch = []
    for path, _, _ in rows:
        with Image.open(image_dir / path) as im:
            batch.append(transform(im.convert("RGB")))
    return torch.stack(batch).cuda()


from math import ceil
print(len(jobs), "training runs;", len(jobs) * len(clip_epochs), "epoch measurements")
print(sum(max(clip_epochs) * ceil(j["train_n"] / j["clip_batch"]) for j in jobs), "planned CLIP updates")

## 2. Prepare frozen inputs
Score the adaptation pool with CLIP CDE and extract frozen DINO features once. Inference batch size is separate from training batch sizes. Move the encoders off GPU when not needed.

In [ ]:
model, width, preprocess = clip.load(clip_backbone, device="cuda", download_root=str(root / "weights"))
model.float().eval()
descriptions = json.loads((upstream / f"descriptions/generic/{dataset_name}.json").read_text())
templates = ds_specific_templates[dataset_name]

weights = []
with torch.no_grad():
    for name in names:
        texts = descriptions[name] + [t.format(name) for t in templates]
        tokens = clip.tokenize(texts).cuda()
        vectors = F.normalize(model.encode_text(tokens), dim=1)
        weights.append(F.normalize(vectors.mean(dim=0), dim=0))
cde = torch.stack(weights, dim=1)
print("CDE weights [features, classes]:", tuple(cde.shape))

@torch.no_grad()
def clip_scores(rows):
    scores = []
    for i in range(0, len(rows), inference_batch):
        features = F.normalize(model.encode_image(load_images(rows[i:i+inference_batch], te_transform)), dim=1)
        scores.append((model.logit_scale.exp() * (features @ cde)).softmax(dim=1).cpu())
    return torch.cat(scores)


pool_probs, val_probs = clip_scores(pool), clip_scores(val)
cde_correct = int((val_probs.argmax(1) == y_val).sum())
print(f"CDE baseline: {cde_correct}/{val_n}")

In [ ]:
model.cpu()
torch.cuda.empty_cache()
nola_utils = sys.modules.pop("utils")
try:
    dino = torch.hub.load(str(root / "dino-source"), "dino_vitb16", source="local", pretrained=False)
finally:
    sys.modules["utils"] = nola_utils
state = torch.load(root / "weights/dino_vitbase16_pretrain.pth", map_location="cpu", weights_only=True)
dino.load_state_dict(state)
del state
dino.eval().requires_grad_(False).cuda()


@torch.no_grad()
def dino_features(rows):
    return torch.cat([dino(load_images(rows[i:i+inference_batch], te_transform)).cpu()
                      for i in range(0, len(rows), inference_batch)])



pool_features = dino_features(pool)
val_features = dino_features(val)
del dino
torch.cuda.empty_cache()
model.cuda()
print("Cached DINO features:", tuple(pool_features.shape), tuple(val_features.shape))

## 3. Define one configuration
Keep the upstream selector. Identical DINO-head fits are reused within the kernel; each CLIP run resets the trainable visual normalization weights and creates fresh prompts, classifier and optimizer. Independent random generators keep image shuffling matched across CLIP batch and prompt settings.

In [ ]:
for name, param in model.named_parameters():
    param.requires_grad_("visual" in name and ("ln" in name or "bn" in name))
trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
initial_visual = [p.detach().cpu().clone() for _, p in trainable]
hidden = model.visual.class_embedding.numel()
bound = (6 / (3 * 16 * 16 + hidden)) ** 0.5


def strong(image):
    return tr_transforms(image)[1]


import time
from functools import lru_cache
from contextlib import redirect_stdout
from io import StringIO

@lru_cache(None)
def fit_head(n, k, batch, epochs):
    with redirect_stdout(StringIO()):
        ids, _, labels = top_k_indices_per_class2(dict(total_emb=pool_probs[:n], idxs=torch.arange(n), labels=audit[:n]), k)
    torch.manual_seed(seed)
    order = torch.Generator().manual_seed(seed)
    head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, len(names)))
    opt = torch.optim.Adam(head.parameters(), lr=head_lr)
    for _ in range(epochs):
        for rows in torch.randperm(len(ids), generator=order).split(batch):
            opt.zero_grad()
            F.cross_entropy(head(pool_features[ids[rows]]), labels[rows]).backward()
            opt.step()
    with torch.no_grad():
        targets = head(pool_features[:n]).argmax(1)
        correct = int((head(val_features).argmax(1) == y_val).sum())
    stats = dict(anchors=len(ids), classes_with_anchors=labels.unique().numel(),
                 anchor_correct=int((audit[ids] == labels).sum()), dino_head=correct,
                 head_updates=epochs * ceil(len(ids) / batch))
    return targets, stats


def run_clip(j):
    n, b = j["train_n"], j["clip_batch"]
    try:
        targets, stats = fit_head(n, j["anchor_cap"], j["head_batch"], j["head_epochs"])
    except NotImplementedError:
        return [dict(status="selector found no top-3 candidate for a class")]
    with torch.no_grad():
        for (_, param), initial in zip(trainable, initial_visual):
            param.copy_(initial)
    random.seed(seed)
    torch.manual_seed(seed)
    order = torch.Generator().manual_seed(seed)
    prompt = nn.Parameter(torch.empty(1, j["prompts"], hidden, device="cuda").uniform_(-bound, bound))
    classifier = nn.Linear(width, len(names), bias=False).cuda()
    with torch.no_grad():
        classifier.weight.copy_(cde.T)

    def logits(images):
        tokens = model.visual.embeddings_patch(images)
        tokens = torch.cat((tokens[:, :1], prompt.expand(len(images), -1, -1), tokens[:, 1:]), dim=1)
        return classifier(model.visual.forward_after_patch_embeddings(tokens))

    opt = torch.optim.AdamW([
        {"params": [prompt, *classifier.parameters(), *(p for name,p in trainable if "bias" not in name)]},
        {"params": [p for name,p in trainable if "bias" in name], "weight_decay": 0.0}], lr=j["clip_lr"])
    records = []
    start = last_report = time.monotonic()
    updates = 0
    planned = max(clip_epochs) * ceil(n / b)
    for epoch in range(1, max(clip_epochs)+1):
        for ids in torch.randperm(n, generator=order).split(b):
            images = load_images([pool[i] for i in ids.tolist()], strong)
            opt.zero_grad()
            F.cross_entropy(logits(images), targets[ids].cuda(), label_smoothing=clip_label_smoothing).backward()
            opt.step()
            updates += 1
            now = time.monotonic()
            if now - last_report >= 300:
                remaining_mins = (now - start) * (planned-updates) / updates / 60
                print(f"  {updates}/{planned} updates; current run ETA about {remaining_mins:.0f} min", flush=True)
                last_report = now
        if epoch in clip_epochs:
            with torch.no_grad():
                pred = torch.cat([logits(load_images(val[i:i+inference_batch], te_transform)).argmax(1).cpu()
                                  for i in range(0, val_n, inference_batch)])
            correct = int((pred == y_val).sum())
            records.append(dict(stats, status="ok", clip_epoch=epoch, clip_updates=epoch*ceil(n/b),
                                cde=cde_correct, nola=correct, delta_cde=correct-cde_correct,
                                delta_dino=correct-stats["dino_head"], val_n=val_n))
    return records

## 4. Run and read the table
A JSON line is saved per finished configuration with all epoch measurements. Rerun this cell to resume; an interrupted configuration restarts, not its completed neighbors. Change the results path if you change the method or transforms. The ETA extrapolates time per update, so it is rough until several settings finish. `delta_cde` and `delta_dino` compare with the two validation baselines; compare matched configurations and confirm the selected recipe separately on held-out test data.

In [ ]:
import time

saved = [json.loads(line) for line in results_path.read_text().splitlines()] if results_path.exists() else []
protocol = dict(seed=seed, val_seed=val_seed, val_n=val_n, clip_epochs=sorted(clip_epochs), pool_n=len(pool))
# Preserve compatibility with earlier EuroSAT results at these unchanged defaults.
if (clip_backbone, head_lr, clip_label_smoothing) != ("ViT-B/32", 0.001, 0.1):
    protocol.update(backbone=clip_backbone, head_lr=head_lr, smoothing=clip_label_smoothing)
active = [entry for entry in saved if entry["protocol"] == protocol and entry["job"] in jobs]
pending = [j for j in jobs if j not in [entry["job"] for entry in active]]

def budget(j):
    return max(clip_epochs) * ceil(j["train_n"] / j["clip_batch"])

for j in pending[:run_limit]:
    print(f"{len(active)+1}/{len(jobs)}: {j}", flush=True)
    start = time.monotonic()
    records = run_clip(j)
    entry = dict(protocol=protocol, job=j, seconds=round(time.monotonic()-start, 2), records=records)
    with results_path.open("a") as f:
        f.write(json.dumps(entry) + "\n")
    active.append(entry)
    successful = [e for e in active if e["records"][0]["status"] == "ok"]
    remaining = [x for x in jobs if x not in [e["job"] for e in active]]
    if successful:
        seconds_per_update = sum(e["seconds"] for e in successful) / sum(budget(e["job"]) for e in successful)
        print(f"Completed in {entry['seconds']:.1f}s; {len(remaining)} left; rough full-grid remainder {seconds_per_update * sum(map(budget, remaining))/3600:.1f}h", flush=True)
    else:
        print(records[0]["status"], flush=True)

results = pd.DataFrame([{**e["job"], **row} for e in active for row in e["records"]])
display(results.sort_values("nola", ascending=False) if "nola" in results else results)

## 5. Analyze hyperparameters (no training)
Run this cell after §1, even in a fresh kernel without running the models. It reads the saved JSONL. The first graph shows stage accuracy and prompted CLIP versus update count. The second compares the higher minus lower value of one parameter at a time, matching every other setting and the evaluation split. Points are individual matched comparisons; the diamond is their mean, **not** a confidence interval. A blank row is untested. Epoch, batch, and data-size effects also change how many optimizer updates are taken; inspect `clip_updates` before attributing a difference to one mechanism. Early results cover only the settings that finished.

In [ ]:
import matplotlib.pyplot as plt

analysis_protocol = dict(seed=seed, val_seed=val_seed, val_n=val_n, clip_epochs=sorted(clip_epochs), pool_n=len(pool))
if (clip_backbone, head_lr, clip_label_smoothing) != ("ViT-B/32", 0.001, 0.1):
    analysis_protocol.update(backbone=clip_backbone, head_lr=head_lr, smoothing=clip_label_smoothing)
saved_rows = [json.loads(line) for line in results_path.read_text().splitlines()] if results_path.exists() else []
finished = {tuple(e["job"].get(k) for k in grid): e for e in saved_rows
            if e["protocol"] == analysis_protocol and e["job"] in jobs}
a = pd.DataFrame([{**e["job"], **row} for e in finished.values() for row in e["records"] if row["status"] == "ok"])
if a.empty:
    print("No completed settings match this grid and validation protocol yet.")
else:
    a = a.assign(nola_pct=100*a.nola/a.val_n, cde_pct=100*a.cde/a.val_n, dino_pct=100*a.dino_head/a.val_n)
    axes = [*grid, "clip_epoch"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
    for epoch, g in a.groupby("clip_epoch"):
        left.scatter(g.clip_updates, g.nola_pct, label=f"NoLA epoch {epoch}", alpha=.8)
    left.axhline(a.cde_pct.iloc[0], color="black", linestyle="--", label="CDE baseline")
    left.set(xlabel="CLIP optimizer updates", ylabel="Validation accuracy (%)", title="NoLA vs. training budget")
    left.legend()
    for label, col in [("CDE", "cde_pct"), ("DINO head", "dino_pct"), ("NoLA", "nola_pct")]:
        right.scatter([label]*len(a), a[col], alpha=.55)
    right.set(ylabel="Validation accuracy (%)", title="All completed measurements by stage")
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    labels, summaries = [], []
    for i, key in enumerate(axes):
        values = sorted(clip_epochs if key == "clip_epoch" else grid[key])
        for lo, hi in zip(values, values[1:]):
            wide = a.pivot(index=[k for k in axes if k != key], columns=key, values="nola_pct")
            delta = (wide[hi]-wide[lo]).dropna() if {lo,hi}.issubset(wide.columns) else pd.Series(dtype=float)
            position = len(labels)
            labels.append(f"{key}: {lo} → {hi} (n={len(delta)})")
            if len(delta):
                ax.scatter(delta, [position]*len(delta), alpha=.5, color="tab:blue")
                ax.scatter(delta.mean(), position, marker="D", color="black")
                summaries.append(dict(parameter=key, low=lo, high=hi, matched_pairs=len(delta),
                                      mean_change_pp=delta.mean(), min_pp=delta.min(), max_pp=delta.max()))
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.invert_yaxis()
    ax.axvline(0, color="grey", linestyle="--")
    ax.set(xlabel="Higher value minus lower value (validation percentage points)", title="Matched one-parameter changes")
    plt.show()
    print(f"{len(finished)}/{len(jobs)} jobs logged; {len(a)} usable epoch measurements")
    if summaries:
        display(pd.DataFrame(summaries).round({"mean_change_pp": 2, "min_pp": 2, "max_pp": 2}))

### Reading the authors’ defaults against the grid

- **16 visual prompts** is the executable default and the supplement's value. Compare 16 vs 50 on matched settings; the default does not prove either is optimal here.
- **Anchor cap** is `max(16, floor(0.2 * train_n / classes))` in the release, with a further cap of 512 in the paper. The 4 vs 16 comparison asks whether adding more anchors at these small data sizes improves coverage enough to offset noise. Look at anchor count, correct anchors and DINO accuracy, not just NoLA.
- **DINO head** uses batch 32 and 50 epochs in the release. Batch 32 is covered; 3 and 20 epochs are local comparison points, not a test of 50. At equal epochs, bigger batches give fewer updates.
- **Prompted CLIP** uses batch 512 and up to 50 epochs in the release. Batches 4 and 8 are local limits, not evidence about 512. The release uses AdamW LR 0.004; the [preprint](https://arxiv.org/html/2411.19346v2) prose says AdamW at 0.002 and the supplement lists Adam at 0.004. Our LR axis compares values with the same AdamW optimizer.
- Increasing **training size, epoch count, or batch size** changes the data seen per update or the total update count. Log both. Multiple seeds or fresh validation are needed before claiming a reliable default change.